# grahspj: NGC 5548 Joint SED + SDSS Spectrum Fit

This notebook uses Bandwagon to collect broadband photometry and discover SDSS spectra for NGC 5548, then fits both data sets with one `grahspj` model. The joint model shares the grahspj host and AGN continuum with the spectroscopic likelihood. Broadband photometry keeps grahspj's native SED-scale AGN features, while the SDSS spectral likelihood adds jaxqsofit's detailed tied-line, Fe II, and Balmer components on top of the shared continuum.


In [ ]:
from pathlib import Path
import inspect
import sys

import astropy.units as u
from astropy.coordinates import SkyCoord
from astroquery.sdss import SDSS
import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().resolve()
if not (repo_root / "src" / "grahspj").is_dir():
    repo_root = repo_root.parent
bandwagon_root = repo_root.parent / "bandwagon"
jaxqsofit_root = repo_root.parent / "jaxqsofit"
for src_root in (repo_root / "src", bandwagon_root / "src", jaxqsofit_root / "src"):
    if src_root.is_dir() and str(src_root) not in sys.path:
        sys.path.insert(0, str(src_root))

# Notebook kernels keep imported modules alive between edits. Drop grahspj and
# jaxqsofit so rerunning this cell always uses the sibling source trees above.
for module_name in list(sys.modules):
    if module_name == "grahspj" or module_name.startswith("grahspj."):
        del sys.modules[module_name]
    if module_name == "jaxqsofit" or module_name.startswith("jaxqsofit."):
        del sys.modules[module_name]

from bandwagon import matches_to_photometry, query_archival_spectra, xmatch_catalogs
from grahspj import (
    AGNConfig,
    FilterSet,
    FitConfig,
    GRAHSPJ,
    GalaxyConfig,
    InferenceConfig,
    JaxQSOFitConfig,
    LikelihoodConfig,
    Observation,
    PhotometryData,
    SpectroscopyConfig,
    SpectroscopyData,
)

if "backend" not in inspect.signature(SpectroscopyConfig).parameters:
    raise RuntimeError(
        "This notebook imported an older grahspj.SpectroscopyConfig without "
        "the backend option. Restart the kernel or rerun this import cell. "
        f"Imported grahspj from: {sys.modules['grahspj'].__file__}"
    )


In [ ]:
OBJECT_NAME = "NGC 5548"
DSPS_SSP_FN = repo_root.parent / "jaxqsofit" / "tempdata.h5"

coord = SkyCoord.from_name(OBJECT_NAME)
coord


## Query Bandwagon Photometry


In [ ]:
matches = xmatch_catalogs(coord, source_id=[OBJECT_NAME])
for name, table in matches.items():
    print(f"{name}: {len(table)} matched rows")

phot = matches_to_photometry(matches, max_mag_err=1.0)
phot = phot[np.isfinite(phot["flux_mjy"]) & np.isfinite(phot["flux_err_mjy"]) & (phot["flux_err_mjy"] > 0)]
phot.sort("match_distance_arcsec")
phot["catalog", "filter_name", "speclite_name", "flux_mjy", "flux_err_mjy", "psf_fwhm_arcsec", "match_distance_arcsec"]


## Query Bandwagon SDSS Spectra


In [ ]:
spectra = query_archival_spectra(
    coord,
    source_id=[OBJECT_NAME],
    providers=("sdss",),
    radius_arcsec=5.0,
)
spectra.sort("redshift")
spectra[
    "provider",
    "release",
    "obs_id",
    "redshift",
    "quality",
    "sdss_plate",
    "sdss_mjd",
    "sdss_fiberid",
]


In [ ]:
sdss_rows = spectra[spectra["provider"] == "sdss"]
if len(sdss_rows) == 0:
    raise RuntimeError("No SDSS spectrum was found for NGC 5548.")

# Prefer rows with clean SDSS zwarning when available.
quality = np.asarray([str(q).strip() for q in sdss_rows["quality"]])
clean = np.flatnonzero((quality == "0") | (quality == ""))
row = sdss_rows[int(clean[0])] if clean.size else sdss_rows[0]

plate = int(row["sdss_plate"])
mjd = int(row["sdss_mjd"])
fiberid = int(row["sdss_fiberid"])
redshift = float(row["redshift"])
print(f"Using SDSS spectrum plate={plate}, mjd={mjd}, fiber={fiberid}, z={redshift:.5f}")

sdss_hdul = SDSS.get_spectra(plate=plate, mjd=mjd, fiberID=fiberid)[0]
sdss_data = sdss_hdul[1].data
wave_obs = 10.0 ** np.asarray(sdss_data["loglam"], dtype=float)
flux_1e17 = np.asarray(sdss_data["flux"], dtype=float)
ivar = np.asarray(sdss_data["ivar"], dtype=float)
err_1e17 = np.where(ivar > 0.0, 1.0 / np.sqrt(ivar), np.nan)


In [ ]:
C_ANG_PER_S = 2.99792458e18

def flambda_1e17_to_mjy(wave_angstrom, flux_1e17):
    f_lambda_cgs = np.asarray(flux_1e17, dtype=float) * 1.0e-17
    f_nu_cgs = f_lambda_cgs * np.asarray(wave_angstrom, dtype=float) ** 2 / C_ANG_PER_S
    return f_nu_cgs / 1.0e-26


spec_flux_mjy = flambda_1e17_to_mjy(wave_obs, flux_1e17)
spec_err_mjy = flambda_1e17_to_mjy(wave_obs, err_1e17)
spec_mask = (
    np.isfinite(wave_obs)
    & np.isfinite(spec_flux_mjy)
    & np.isfinite(spec_err_mjy)
    & (spec_err_mjy > 0.0)
    & (wave_obs > 3600.0)
    & (wave_obs < 9000.0)
)

plt.figure(figsize=(8, 3))
plt.plot(wave_obs[spec_mask], spec_flux_mjy[spec_mask], lw=0.6)
plt.xlabel("Observed wavelength [Angstrom]")
plt.ylabel("Flux density [mJy]")
plt.title(f"{OBJECT_NAME} SDSS spectrum")
plt.tight_layout()


In [ ]:
spec_wave_native = wave_obs[spec_mask]
spec_flux_native = spec_flux_mjy[spec_mask]
spec_err_native = spec_err_mjy[spec_mask]

print(f"Using {spec_wave_native.size:,} native SDSS spectral pixels")



## Build A Joint Photometry + Spectrum Config


In [ ]:
filter_names = [str(name) for name in phot["filter_name"]]
speclite_names = {str(row["filter_name"]): str(row["speclite_name"]) for row in phot}

cfg = FitConfig(
    observation=Observation(
        object_id="NGC5548_joint",
        redshift=redshift,
        fit_redshift=False,
        ra=float(coord.ra.deg),
        dec=float(coord.dec.deg),
        apply_mw_deredden=False,
    ),
    photometry=PhotometryData(
        filter_names=filter_names,
        fluxes=[float(x) for x in phot["flux_mjy"]],
        errors=[float(x) for x in phot["flux_err_mjy"]],
        is_upper_limit=[False] * len(phot),
        psf_fwhm_arcsec=[float(x) if np.isfinite(x) else None for x in phot["psf_fwhm_arcsec"]],
    ),
    filters=FilterSet(speclite_names=speclite_names, use_grahsp_database=False),
    spectroscopy=SpectroscopyData(
        wave_obs=spec_wave_native.tolist(),
        fluxes=spec_flux_native.tolist(),
        errors=spec_err_native.tolist(),
        mask=[True] * len(spec_wave_native),
        instrument="SDSS",
        aperture_diameter_arcsec=3.0,
    ),
    spectroscopy_config=SpectroscopyConfig(
        enabled=True,
        backend="jaxqsofit",
        fit_scale=True,
        scale_prior_sigma_dex=0.4,
        systematics_width=0.08,
        student_t_df=5.0,
        # These jaxqsofit switches affect only the spectroscopic likelihood.
        # Photometric fluxes still use grahspj native SED-scale AGN features.
        jaxqsofit=JaxQSOFitConfig(
            use_spectral_lines=True,
            use_tied_lines=True,
            use_spectral_smart_priors=True,
            use_spectral_feii=True,
            use_spectral_balmer_continuum=True,
            line_flux_scale_mjy=0.1,
        ),
    ),
    galaxy=GalaxyConfig(
        dsps_ssp_fn=str(DSPS_SSP_FN),
        n_wave=2048,
        rest_wave_min=100.0,
        rest_wave_max=3.0e6,
    ),
    agn=AGNConfig(
        agn_type=1,
        line_width_kms_default=3000.0,
        lines_strength_default=1.0,
        feii_strength_default=1.0,
        fit_balmer_continuum=True,
        balmer_continuum_default=0.1,
    ),
    likelihood=LikelihoodConfig(
        use_host_capture_model=True,
        systematics_width=0.08,
        fit_intrinsic_scatter=True,
    ),
    inference=InferenceConfig(map_steps=600, learning_rate=5.0e-3, seed=3),
)
cfg.validate()
cfg


## Fit

The first run below uses MAP/SVI with staged continuum-first initialization, which is now the default for `fit_method="optax"`. Increase `optax_steps` or switch to `fit_method="optax+nuts"` once the model and data choices look sensible.


In [ ]:
fitter = GRAHSPJ(cfg)
result = fitter.fit(
    fit_method="optax",
    optax_steps=cfg.inference.map_steps,
    optax_lr=cfg.inference.learning_rate,
    progress_bar=True,
    plot_fig=False,
    save_fig=False,
)
result["summary"]


In [ ]:
pred = fitter.predict()
print(f"Joint backend: {cfg.spectroscopy_config.backend}")
print("Spectra in model context:", len(fitter.context.spec_instruments))
print("Spectrum instruments:", fitter.context.spec_instruments)
print("Predictive spectral sites:", sorted(k for k in pred if k.startswith("jqf_") or k.startswith("pred_spectrum")))


In [ ]:
fig = fitter.plot_jaxqsofit_spectrum(
    spectrum_index=0,
    show_plot=True,
    plot_residual=True,
    plot_legend=True,
)


In [ ]:
fitter.plot_sed()